In [1]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    temperature=0,
    groq_api_key="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",
    model="meta-llama/llama-4-maverick-17b-128e-instruct"    
)
response=llm.invoke("who defeated  rocks d xebec and who were his crew members in one piece ")
print(response.content);

## Rocks D. Xebec and His Crew

Rocks D. Xebec is a powerful pirate from the One Piece universe. According to the One Piece Wiki, he was the captain of the Rocks Pirates.

### Defeat of Rocks D. Xebec

Rocks D. Xebec was defeated by **Monkey D. Garp** and **Gol D. Roger** in a joint effort. This significant event occurred around 38 years ago, before the main storyline of the One Piece series.

### Crew Members of the Rocks Pirates

The known crew members of the Rocks Pirates include:

1. **Rocks D. Xebec** (Captain)
2. **Charlotte Linlin (Big Mom)**
3. **Kaido**
4. **Marshall D. Teach (Blackbeard)**
5. **Silvers Rayleigh** (not officially a member, but associated with the crew)
6. **Ochibi** (also known as "Little Oars" or just "Oars")

These crew members were part of the Rocks Pirates, one of the most feared pirate crews in the One Piece world.


In [38]:
from langchain_community.document_loaders import WebBaseLoader
loader =WebBaseLoader(" https://www.accenture.com/in-en/careers/jobdetails?id=AIOC-S01542976_en&title=Web%2BDeveloper%2BNew%2BAssociate ")
page_data=loader.load().pop().page_content
print(page_data)






Web Developer New Associate


























































































































Skip to main content
Skip to footer













Menu




 
Accenture


 


Accenture








Close Menu





What we do

 





Back



What we do





Capabilities
Capabilities



Cloud 
					


Customer Service 
					


Cybersecurity 
					


Data and Artificial Intelligence 
					


Digital Engineering and Manufacturing 
					


Ecosystem Partners 
					


Emerging Technology 
					


Finance and Risk Management 
					


Infrastructure and Capital Projects 
					


Learning 
					


Managed Services 
					


Marketing and Experience 
					


Metaverse 
					


Sales and Commerce 
					


Strategy 
					


Supply Chain 
					


Sustainability 
					


Talent and Organization 
					


Technology Transformation 
					





Industries
Industries



Aerospace and Defense 
					


Automotive 
					


Banking 
					


Capital Markets 
	

In [39]:
from langchain_core.prompts import PromptTemplate
prompt_extract=PromptTemplate.from_template("""
    ###SCRAPED TEXT FROM WEBSITE:
    {page_data},
    ###INSTRUCTIONS:
    The scraped text is from a job posting site .your job is to extract the  job postings and return them in a json format containing following 
    key roles : 'role ' , 'experience',' skills '   and 'description ' . only return valid JSON .
    ### VALID JSON (NO PREAMBLE) :
""")
chain_extract = prompt_extract | llm
res=chain_extract.invoke(input={"page_data": page_data})
print(res.content)

```json
{
  "role": "Web Developer New Associate",
  "experience": "0-2 years",
  "skills": "HTML (Hypertext Markup Language), Digital Marketing Ads & Promotion creation/design",
  "description": "Help balance increased marketing complexity and diminishing marketing resources. Drive marketing performance with deep functional and technical expertise, while accelerating time-to-market and operating efficiencies at scale through Data and Technology, Next Generation Content Services, Digital Marketing Services & Customer Engagement and Media Growth Services. Role requires working primarily on HTML for developing web pages."
}
```


In [33]:
from langchain_core.output_parsers import JsonOutputParser
json_parser=JsonOutputParser()
json_res=json_parser.parse(res.content)
json_res

{'role': 'Web Developer New Associate',
 'experience': '0-2 years',
 'skills': 'HTML',
 'description': 'Help balance increased marketing complexity and diminishing marketing resources. Drive marketing performance with deep functional and technical expertise, while accelerating time-to-market and operating efficiencies at scale through Data and Technology, Next Generation Content Services, Digital Marketing Services & Customer Engagement and Media Growth Services. Role requires Digital Marketing Ads & Promotion creation/design. You will be responsible for working primarily on HTML (Hypertext Markup Language) for developing web pages.'}

In [24]:
type(json_res)

dict

In [25]:
import pandas as pd
df=pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [34]:
import chromadb 
import uuid
client=chromadb.PersistentClient('vectorstore')
collection=client.get_or_create_collection(name="portfolio")
if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row['Techstack'],
                      metadatas={'links':row['Links']},
                      ids=[str(uuid.uuid4())])

In [35]:
job=json_res
job['skills']

'HTML'

In [28]:
skill=json_res
links=collection.query(query_texts=skill['skills'],n_results=2).get('metadatas',[])
links

[[{'links': 'https://example.com/wordpress-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}]]

In [37]:
prompt_email=PromptTemplate.from_template("""
###JOB DECSRIPTION:
{job_description}
### INSTRUCTION:
    You are Yagnesh , a HR at XYZ Solutions.XYZ Solutions is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of XYZ Solutions 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase XYZ's portfolio: {links}
        Remember you are Yagnesh, HR at XYZ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
""")
chain_email=prompt_email | llm 
email_res=chain_email.invoke({'job_description':str(job),'links':links})
print(email_res.content)

Subject: Expert Web Development Services for Your Digital Marketing Needs

Dear Hiring Manager,

I came across the job description for a Web Developer New Associate at your organization, and I believe XYZ Solutions can be a valuable partner in fulfilling your requirements. Our team has a proven track record of delivering high-quality web development services that cater to the needs of businesses looking to enhance their digital presence.

The job description mentions the need for expertise in HTML for developing web pages, particularly for digital marketing ads and promotions. Our team of skilled web developers has extensive experience in creating responsive and engaging web pages using HTML. We understand the importance of accelerating time-to-market and operating efficiencies at scale, and our solutions are designed to meet these needs.

At XYZ Solutions, we have a portfolio of successful projects that demonstrate our capabilities in web development. You can view some of our notable 